# TPCのゲインマップ：rawツリー → 1-hitスペクトル → ピークフィット → Ka/Kb

`RawTreeWriter_sm3` のFADC波形から、アノードch × カソードchの領域ごとに
**E [keV] = Ka × Q [ADU × sample] + Kb [keV]** を決めます。
SMILE-3の `CalcFADC` は電荷をpCに換算しないので、Kaの単位は **keV/(ADU × sample)** です。
SMILE-2+のpC単位の係数やガス増幅率への換算係数は転用しません。

FADC ch = 4 × board ID + ch（0〜47）= strip / 32。panel = board ID / 3 = FADC ch / 12。
同じパネルの12 × 12組を4パネル、計576組が対象です。
線源の同定とフィット範囲はユーザーが指定します。未知の線源や不明なピークを推測で較正しません。

In [ ]:
# ===== 入力と出力(絶対パス。データ量が大きいので、リポジトリの外の NAS などに置く)=====
from pathlib import Path
# 入力: RawTreeWriter_sm3/raw を含むROOTファイル(同じ測定条件のもの)
RAWS = [
    Path("/mnt/nas_eureka/onoda/<書き換え>/tpc_raw_20261111_run71.root"),
]
# 出力: このノートブックが書くファイルの置き場
OUT = Path("/mnt/nas_eureka/onoda/<書き換え>/tpc_gain_20261111_run71")

for p in [*RAWS, OUT]:
    if "<" in str(p):
        raise ValueError(f"パスを書き換えてください: {p}")
OUT.mkdir(parents=True, exist_ok=True)
print("入力:"); [print("  ", p) for p in RAWS]
print("出力:", OUT)


In [ ]:
import sys, json, subprocess
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from IPython.display import Image, display
import ROOT
ROOT.gROOT.SetBatch(True)
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'run_script/run_chain.py').exists())
SCRIPTS = REPO / 'scripts/calib'
sys.path.insert(0, str(SCRIPTS))
PYTHON = '/usr/bin/python3'  # numpy / PyROOT のある Python
TREE = 'RawTreeWriter_sm3/raw'

## 1. 入力ROOTファイル

このノートブックは **既存の `RawTreeWriter_sm3/raw` を読む**もので、生データを自動で再解析しません。
冒頭の `RAWS` を実際のROOTファイルに変更します。複数ファイルは同じ測定条件のものを指定してください。
run70 は ETCC、run71〜73 は TPC_cal です。測定条件と線源を確認し、モードだけで較正に適しているとは判断しません。
前の積算処理が作った `root_data/20261111_tracks/run*.root` には raw ツリーが無いため、入力にはできません。

入力・出力はデータ量が大きいのでリポジトリの外(NASなど)に置き、冒頭の `RAWS` と `OUT` に絶対パスで書きます。
ROOTファイルがまだ無い場合は、ターミナルで例えば次を実行します。
`python3` は EricaPy をビルドした Python、ノートブックは numpy / PyROOT のある「EricaNext」カーネルを使います。

```bash
source ~/opt/setup_ericanext.sh
cd ~/ANLNext/Erica_git_dev/EricaNext
python3 run_script/run_chain.py smile3_raw --compression 505 \
  -o /mnt/nas_eureka/onoda/<書き換え>/tpc_raw_20261111_run71.root \
  --data-dir ~/QL_PC/storage_g0/20261111_run71 \
  --extra-data-dir ~/QL_PC/storage_g1/20261111_run71
```

`smile3_raw` は既定で `save_hits=true`, `save_fadc=true`, `save_threshold=0.0` です。
波形を閾値で間引いたファイルでは小さい信号が欠けるので、ゲイン較正には間引かないものを使います。

In [ ]:
for path in RAWS:
    if not path.is_file(): raise FileNotFoundError(path)
    f = ROOT.TFile.Open(str(path))
    if not f or f.IsZombie(): raise RuntimeError(f'ROOTファイルを開けません: {path}')
    t = f.Get(TREE)
    if not t: raise RuntimeError(f'{path}: {TREE} がありません')
    required = ['fadc_ch', 'fadc_wave']
    missing = [b for b in required if not t.GetBranch(b)]
    if missing: raise RuntimeError(f'{path}: 必要なブランチがありません: {missing}')
    print(path.name, t.GetEntries(), 'events')
    f.Close()

## 2. 電荷の計算と1-hitスペクトル

`scripts/calib/make_tpc_spec1hit.py` で `CalcFADC` と同じ積分を行います。
ベースラインは最初の30 sampleの平均、積分範囲はその後から波形末尾まで。
ベースラインから20 ADUを超えるsampleだけを足します（1 sample = 20 ns）。
カソード波形はrawツリーですでに反転済みです。

選別は「アノード全体で電荷のあるchが1本、**同じパネル内**のカソードで1本」です。
他パネルのカソードに電荷があるイベントも許す、既存スクリプトの定義です。
保存閾値は0を推奨します。異なる閾値・測定条件のスペクトルは混ぜません。
再実行時もスペクトルを作り直すため、入力変更時に古いcacheを再利用しません。

In [ ]:
BASE_WINDOW = 30   # ベースラインを取る長さ [sample]
THRESHOLD = 20.0   # ベースラインからの積分閾値 [ADU]
for path in RAWS:
    f = ROOT.TFile.Open(str(path))
    info = f.Get('RawTreeWriter_sm3/run_info')
    if not info or not info.GetEntries(): raise RuntimeError(f'{path}: run_infoがありません')
    info.GetEntry(0)
    if info.save_threshold != 0: raise RuntimeError('ゲイン較正にはsave_threshold=0のrawを用意してください')
    if not 0 < BASE_WINDOW < info.n_fadc_clk: raise ValueError('BASE_WINDOWが不正です')
    f.Close()
SPEC_FILE = OUT / 'spec1hit.root'
cmd = [PYTHON, str(SCRIPTS / 'make_tpc_spec1hit.py'), '-o', str(SPEC_FILE), *map(str, RAWS),
       '--base-window', str(BASE_WINDOW), '--threshold', str(THRESHOLD), '--overwrite']
print(' '.join(cmd)); subprocess.run(cmd, check=True)
display(Image(filename=str(OUT / 'qa/spec1hit_board.png')))

## 3. 組ごとの統計とスペクトル

1-hitスペクトルは2000ビン、0〜10000 ADU × sample（5 ADU × sample/bin）です。
ピークが軸外ならスペクトル作成スクリプトの範囲を見直し、端で切れた山をフィットしません。
まず統計マップと任意の組のスペクトルを見て、線源に対応する2本のピークを確認します。

In [ ]:
f = ROOT.TFile.Open(str(SPEC_FILE))
spectra = {}
for a in range(48):
    for c in range(12*(a//12), 12*(a//12+1)):
        h = f.Get(f'Anode_1Hit_A{a:02d}_C{c:02d}')
        if h:
            spectra[a,c] = h.Clone(f'gain_A{a}_C{c}'); spectra[a,c].SetDirectory(0)
f.Close()
counts = np.full((48,48), np.nan)
for (a,c), h in spectra.items(): counts[c,a] = h.Integral()
fig, ax = plt.subplots(figsize=(9,7))
m = ax.imshow(np.ma.masked_invalid(np.ma.masked_less(counts,1)), origin='lower',
              norm=LogNorm(vmin=1, vmax=max(2, np.nanmax(counts) if spectra else 2)))
fig.colorbar(m, ax=ax, label='1-hit events'); ax.set(xlabel='Anode FADC ch', ylabel='Cathode FADC ch')
fig.tight_layout(); fig.savefig(OUT / 'counts.png', dpi=150); plt.show()
print('スペクトルのある組:', len(spectra))

In [ ]:
PAIR = next(iter(spectra), (0,0))  # 表示したい (anode ch, cathode ch)
h = spectra.get(PAIR)
if h:
    x = np.array([h.GetBinCenter(i) for i in range(1,h.GetNbinsX()+1)])
    y = np.array([h.GetBinContent(i) for i in range(1,h.GetNbinsX()+1)])
    plt.figure(figsize=(10,4)); plt.step(x,y,where='mid')
    plt.xlabel('Anode charge [ADU x sample]'); plt.ylabel('Events / 5 ADU x sample')
    plt.title(f'A{PAIR[0]} C{PAIR[1]}'); plt.show()

## 4. 線のエネルギーとフィット範囲を設定

`ENERGIES_KEV` に **測定で確認した2本の線**を小さい順に指定します。
例えばBa/CuのX線と確認できた測定なら `[8.04, 30.85]` keVですが、今回のrunの線源を意味しません。
`WINDOWS[(a,c)] = [(低エネルギー線のQ下限,Q上限), (高エネルギー線のQ下限,Q上限)]` を指定します。
`DEFAULT_WINDOWS` を指定すれば、個別指定のない全組に同じ範囲を適用します。
位置依存性が大きい場合は個別に指定してください。未設定のままでは較正ファイルを作りません。

背景込みのガウス（gaus + pol1）を指定範囲でフィットします。
統計、fit status、自由度、χ²/ndf、中心・幅の制限への張り付き、中心誤差、有意度を確認します。
有意度はガウスの高さ / その誤差で、背景と強く相関する場合は厳しい判定になります。
判定に通った2点を結ぶ直線でKa/Kbを求めます（原点は追加しません）。

In [ ]:
ENERGIES_KEV = [8.04, 30.85]  # 例: [8.04, 30.85]。線源を確認して設定 [keV]
DEFAULT_WINDOWS = None  # 例: [(Q1下限,Q1上限), (Q2下限,Q2上限)] [ADU x sample]
WINDOWS = {}  # {(anode_ch, cathode_ch): [(Q1下限,Q1上限), (Q2下限,Q2上限)]}
MIN_EVENTS = 200      # 組ごとの最小1-hitイベント数 [events]
FIT_REBIN = 2         # フィット時にまとめる元ビン数 [bins]（1なら5 ADU x sample/bin）
CHI2NDF_MAX = 5.0     # χ²/ndf上限 [dimensionless]
SIGNIFICANCE_MIN = 3.0  # ガウス高さ/誤差の下限 [dimensionless]

In [ ]:
def fit_line(h, window, name):
    lo, hi = map(float, window)
    if not h.GetXaxis().GetXmin() < lo < hi < h.GetXaxis().GetXmax():
        raise ValueError(f'{name}: fit範囲がスペクトルの軸範囲外です')
    t = h.Clone(name); t.SetDirectory(0); t.Rebin(FIT_REBIN)
    t.GetXaxis().SetRangeUser(lo, hi)
    seed = t.GetBinCenter(t.GetMaximumBin())
    width = hi-lo
    bg = max(0, (t.GetBinContent(t.FindBin(lo))+t.GetBinContent(t.FindBin(hi)))/2)
    fn = ROOT.TF1(name+'_fit', 'gaus(0)+pol1(3)', lo, hi)
    fn.SetParameters(max(1,t.GetMaximum()-bg), seed, width/8, bg, 0)
    fn.SetParLimits(0, 0, max(10,10*t.GetMaximum()))
    fn.SetParLimits(1, lo, hi); fn.SetParLimits(2, t.GetBinWidth(1)/2, width/2)
    status = int(t.Fit(fn, 'QSN0R'))
    mu, err, sigma = fn.GetParameter(1), fn.GetParError(1), abs(fn.GetParameter(2))
    chi = fn.GetChisquare()/fn.GetNDF() if fn.GetNDF() > 0 else np.inf
    sig = fn.GetParameter(0)/fn.GetParError(0) if fn.GetParError(0)>0 else 0
    reasons = []
    if status != 0: reasons.append(f'fit status {status}')
    if not np.isfinite(mu) or not lo+0.02*width < mu < hi-0.02*width: reasons.append('中心が範囲端')
    if not t.GetBinWidth(1)*0.51 < sigma < width*0.49: reasons.append('幅が制限端')
    if not 0 < err < abs(mu)*0.1: reasons.append('中心誤差が大きい')
    if not chi < CHI2NDF_MAX: reasons.append('chi2/ndf')
    if not sig >= SIGNIFICANCE_MIN: reasons.append('有意度不足')
    return dict(mu=mu, error=err, sigma=sigma, chi2ndf=chi, significance=sig,
                reasons=reasons, fn=fn, window=(lo,hi))

if ENERGIES_KEV is None or len(ENERGIES_KEV)!=2 or not 0 < ENERGIES_KEV[0] < ENERGIES_KEV[1]:
    raise ValueError('確認済みの2本の線をENERGIES_KEVに小さい順で設定してください')
if DEFAULT_WINDOWS is None and not WINDOWS:
    raise ValueError('スペクトルを確認してDEFAULT_WINDOWSまたはWINDOWSを設定してください')
assert isinstance(FIT_REBIN, int) and FIT_REBIN > 0 and 2000 % FIT_REBIN == 0
fits, cal, rejected = {}, {}, {}
for pair, h in spectra.items():
    windows = WINDOWS.get(pair, DEFAULT_WINDOWS)
    if windows is None: rejected[pair] = ['fit範囲未指定']; continue
    if h.Integral() < MIN_EVENTS: rejected[pair] = ['統計不足']; continue
    if len(windows)!=2 or not windows[0][1] < windows[1][0]:
        raise ValueError(f'{pair}: 低い線、高い線の順で重ならない2範囲を指定してください')
    r = [fit_line(h,w,f'fit_A{pair[0]}_C{pair[1]}_{k}') for k,w in enumerate(windows)]
    fits[pair] = r
    why = [f'line{k}: {s}' for k,v in enumerate(r) for s in v['reasons']]
    if not r[0]['mu'] < r[1]['mu']: why.append('ピークの順序が不正')
    if why: rejected[pair] = why; continue
    ka = (ENERGIES_KEV[1]-ENERGIES_KEV[0])/(r[1]['mu']-r[0]['mu'])
    kb = ENERGIES_KEV[0]-ka*r[0]['mu']
    if not np.isfinite(ka+kb) or ka<=0: rejected[pair] = ['係数が不正']; continue
    cal[pair] = (ka,kb)
print('較正成功:',len(cal), '/ 576組')
for pair, why in rejected.items(): print(pair, ', '.join(why))

## 5. 各組のフィットと係数マップを確認

フィットした各組のスペクトルを `qa/Axx_Cxx.png` に保存します。
自動判定に落ちた組も図で確認できます。判定を通っても、線を取り違えていないか確認してください。
係数の白い領域は未較正です。マップは物理XYではなくFADC chの組の並びです。

In [ ]:
QA = OUT / 'qa'
QA.mkdir(exist_ok=True)
for pair, rr in fits.items():
    h = spectra[pair]
    x = np.array([h.GetBinCenter(i) for i in range(1,h.GetNbinsX()+1)])
    y = np.array([h.GetBinContent(i) for i in range(1,h.GetNbinsX()+1)])
    fig, ax = plt.subplots(figsize=(8,4))
    ax.step(x,y,where='mid',lw=0.7)
    for k,r in enumerate(rr):
        xx = np.linspace(*r['window'], 200)
        # ROOT fit values are counts per rebinned bin; compare at original bin width.
        ax.plot(xx, [r['fn'].Eval(v)/FIT_REBIN for v in xx], label=f'{ENERGIES_KEV[k]:g} keV')
        ax.axvline(r['mu'], ls=':',color='k')
    ax.set(xlim=(rr[0]['window'][0],rr[1]['window'][1]), xlabel='Anode charge [ADU x sample]',
           ylabel='Events / 5 ADU x sample', title=f'A{pair[0]} C{pair[1]}: '+('OK' if pair in cal else 'NG'))
    ax.legend(); fig.tight_layout(); fig.savefig(QA/f'A{pair[0]:02d}_C{pair[1]:02d}.png',dpi=100); plt.close(fig)
fig, axs = plt.subplots(1,2,figsize=(15,6))
for k, ax in enumerate(axs):
    z = np.full((48,48),np.nan)
    for (a,c), v in cal.items(): z[c,a] = v[k]
    m = ax.imshow(np.ma.masked_invalid(z),origin='lower')
    fig.colorbar(m,ax=ax,label=['Ka [keV/(ADU x sample)]','Kb [keV]'][k])
    ax.set(xlabel='Anode FADC ch',ylabel='Cathode FADC ch',title=['Ka','Kb'][k])
fig.tight_layout(); fig.savefig(OUT/'gain_map.png',dpi=150); plt.show()
if fits:
    pair = next(iter(fits)); display(Image(filename=str(QA/f'A{pair[0]:02d}_C{pair[1]:02d}.png')))

## 6. 保存と適用

未較正の組はKa=Kb=0とし、推測や周囲の平均で埋めません。
ファイルには同じパネルの576組を数値4列 `anode_ch cathode_ch Ka Kb` で保存します。
現行 `TPC_Calibration` の読み込みはコメント行に対応しないため、由来・単位・fit結果は別JSONに保存します。
未較正領域に電荷があると再構成エネルギーが低くなるので、利用時に対象領域を確認してください。

**現行のTPC_CalibrationはSMILE-2+の「同時計数100超」「端ストリップを除く」を適用しません。**
このチュートリアルでもその条件は追加しません。
出力は冒頭の `OUT` で、既存較正ファイルを自動で置換しません。

In [ ]:
if not cal: raise RuntimeError('較正できた組がないためゲインマップを書きません')
CAL = OUT / 'gain_map.dat'
with CAL.open('w') as stream:
    for c in range(48):
        for a in range(12*(c//12),12*(c//12+1)):
            ka,kb = cal.get((a,c),(0.0,0.0))
            stream.write(f'{a} {c} {ka:.12g} {kb:.12g}\n')
report = {'inputs':[str(p.resolve()) for p in RAWS], 'energies_keV':ENERGIES_KEV,
          'charge_unit':'ADU x sample', 'Ka_unit':'keV/(ADU x sample)', 'Kb_unit':'keV',
          'base_window_samples':BASE_WINDOW, 'threshold_ADU':THRESHOLD,
          'default_windows_charge':DEFAULT_WINDOWS,
          'windows_charge':{f'{a},{c}':v for (a,c),v in WINDOWS.items()},
          'fit_rebin':FIT_REBIN, 'min_events':MIN_EVENTS, 'chi2ndf_max':CHI2NDF_MAX,
          'significance_min':SIGNIFICANCE_MIN,
          'calibrated':{f'{a},{c}':list(v) for (a,c),v in cal.items()},
          'rejected':{f'{a},{c}':v for (a,c),v in rejected.items()},
          'fits':{f'{a},{c}':[{k:v for k,v in r.items() if k!='fn'} for r in rr] for (a,c),rr in fits.items()}}
(OUT/'gain_map.json').write_text(json.dumps(report,ensure_ascii=False,indent=2)+'\n')
print('wrote',CAL, f'({len(cal)}/576 calibrated)')

確認後、解析時に `--set TPC_Calibration.NoCalData=false` と
`--set TPC_Calibration.ParameterFile=<gain_map.datの絶対パス>` を指定します。
波形の積分閾値・ベースライン設定は較正時と一致させてください。